# **Step 1**
**Step 1 — Clean Colab setup + COCO download**

In [ ]:
!pip -q install pycocotools opencv-python einops tqdm matplotlib

**Install dependencies**

Imports + GPU check

In [ ]:
import os, random, math
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


**1.1**

Reproducibility (seed)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Seed set to 42")


**1.2**

Project folders

In [ ]:
ROOT = "/content/occlusion_pose_gnn"
COCO_ROOT = f"{ROOT}/data/coco"

os.makedirs(f"{ROOT}/outputs", exist_ok=True)
os.makedirs(COCO_ROOT, exist_ok=True)

print("ROOT:", ROOT)
print("COCO_ROOT:", COCO_ROOT)


In [ ]:
ROOT = "/content/occlusion_pose_gnn"
COCO_ROOT = f"{ROOT}/data/coco"
OUT_DIR = f"{ROOT}/outputs"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(COCO_ROOT, exist_ok=True)

print("ROOT:", ROOT)
print("COCO_ROOT:", COCO_ROOT)
print("OUT_DIR:", OUT_DIR)

**1.3**

Download COCO val2017 + annotations

In [ ]:
%cd $COCO_ROOT

TRAIN_URL = "https://images.cocodataset.org/zips/train2017.zip"
VAL_URL   = "https://images.cocodataset.org/zips/val2017.zip"
ANN_URL   = "https://images.cocodataset.org/annotations/annotations_trainval2017.zip"

def download_if_missing(url, out_name):
    if not os.path.exists(out_name):
        print("Downloading:", out_name)
        !curl -L -k -o $out_name $url
    else:
        print("Already exists:", out_name)

def unzip_if_missing(zip_name, folder_name):
    if not os.path.exists(folder_name):
        print("Extracting:", zip_name)
        !unzip -q $zip_name
    else:
        print("Already extracted:", folder_name)

download_if_missing(TRAIN_URL, "train2017.zip")
download_if_missing(VAL_URL,   "val2017.zip")
download_if_missing(ANN_URL,   "annotations_trainval2017.zip")

unzip_if_missing("train2017.zip", "train2017")
unzip_if_missing("val2017.zip",   "val2017")
unzip_if_missing("annotations_trainval2017.zip", "annotations")

print("\nFolder check:")
print("train2017 exists:", os.path.isdir("train2017"))
print("val2017 exists:  ", os.path.isdir("val2017"))
print("annotations exists:", os.path.isdir("annotations"))
print("train keypoints json exists:",
      os.path.exists("annotations/person_keypoints_train2017.json"))
print("val keypoints json exists:",
      os.path.exists("annotations/person_keypoints_val2017.json"))


**1.4**

Final paths you’ll use later

In [ ]:
TRAIN_IMG_DIR = f"{COCO_ROOT}/train2017"
VAL_IMG_DIR   = f"{COCO_ROOT}/val2017"

TRAIN_ANN_FILE = f"{COCO_ROOT}/annotations/person_keypoints_train2017.json"
VAL_ANN_FILE   = f"{COCO_ROOT}/annotations/person_keypoints_val2017.json"

assert os.path.isdir(TRAIN_IMG_DIR), "train2017 folder missing"
assert os.path.isdir(VAL_IMG_DIR), "val2017 folder missing"
assert os.path.exists(TRAIN_ANN_FILE), "person_keypoints_train2017.json missing"
assert os.path.exists(VAL_ANN_FILE), "person_keypoints_val2017.json missing"

print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TRAIN_ANN_FILE:", TRAIN_ANN_FILE)
print("VAL_IMG_DIR:  ", VAL_IMG_DIR)
print("VAL_ANN_FILE: ", VAL_ANN_FILE)


**1.5**

In [ ]:
from pycocotools.coco import COCO

coco_train = COCO(TRAIN_ANN_FILE)
coco_val   = COCO(VAL_ANN_FILE)

print("Train images:", len(coco_train.imgs), "| Train anns:", len(coco_train.anns))
print("Val images:  ", len(coco_val.imgs),   "| Val anns:  ", len(coco_val.anns))

# **Step 2**

**COCO Dataset + DataLoader + Heatmap Targets**

In [ ]:
COCO_KPT_NAMES = [
    "nose",
    "left_eye", "right_eye",
    "left_ear", "right_ear",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
]
K = len(COCO_KPT_NAMES)

COCO_FLIP_PAIRS = [
    (1,2), (3,4),
    (5,6), (7,8), (9,10),
    (11,12), (13,14), (15,16),
]

# COCO skeleton edges (1-indexed in COCO docs)
COCO_SKELETON = [
    (16,14),(14,12),(17,15),(15,13),(12,13),
    (6,12),(7,13),(6,7),
    (6,8),(7,9),(8,10),(9,11),
    (2,3),(1,2),(1,3),(2,4),(3,5),(4,6),(5,7)
]
COCO_EDGES = [(a-1, b-1) for a,b in COCO_SKELETON]

print("K =", K, "| edges =", len(COCO_EDGES))


**2.2**

Geometry helpers (affine warp + gaussian heatmaps)

In [ ]:
import numpy as np
import cv2

def _get_3rd_point(a, b):
    direct = a - b
    return b + np.array([-direct[1], direct[0]], dtype=np.float32)

def _get_dir(src_point, rot_rad):
    sn, cs = np.sin(rot_rad), np.cos(rot_rad)
    return np.array(
        [src_point[0]*cs - src_point[1]*sn,
         src_point[0]*sn + src_point[1]*cs],
        dtype=np.float32
    )

def get_affine_transform(center, scale, rot_deg, output_size):
    """
    center: (2,) original image coords
    scale: scalar pixel size
    rot_deg: degrees
    output_size: (W,H)
    """
    if isinstance(scale, (float, int)):
        scale = np.array([scale, scale], dtype=np.float32)
    else:
        scale = np.array(scale, dtype=np.float32)

    src_w = scale[0]
    dst_w, dst_h = output_size

    rot_rad = np.pi * rot_deg / 180.0
    src_dir = _get_dir(np.array([0, -0.5*src_w], dtype=np.float32), rot_rad)
    dst_dir = np.array([0, -0.5*dst_w], dtype=np.float32)

    src = np.zeros((3,2), dtype=np.float32)
    dst = np.zeros((3,2), dtype=np.float32)

    src[0] = center
    src[1] = center + src_dir
    src[2] = _get_3rd_point(src[0], src[1])

    dst[0] = np.array([dst_w*0.5, dst_h*0.5], dtype=np.float32)
    dst[1] = dst[0] + dst_dir
    dst[2] = _get_3rd_point(dst[0], dst[1])

    return cv2.getAffineTransform(src, dst)

def affine_transform(pt, trans):
    p = np.array([pt[0], pt[1], 1.0], dtype=np.float32)
    p = trans @ p
    return p[:2]

def draw_gaussian(hm, center, sigma):
    """
    Fully robust gaussian draw (no broadcast errors).
    hm: (H,W)
    center: (x,y) float
    """
    mu_x, mu_y = float(center[0]), float(center[1])
    if not (np.isfinite(mu_x) and np.isfinite(mu_y)):
        return hm

    tmp = int(sigma * 3)
    H, W = hm.shape
    size = 2 * tmp + 1

    # Gaussian grid
    x = np.arange(0, size, 1, np.float32)
    y = x[:, None]
    x0 = y0 = size // 2
    g = np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2))

    # Upper-left and bottom-right corners in hm coordinates
    ul_x = int(np.floor(mu_x - tmp))
    ul_y = int(np.floor(mu_y - tmp))
    br_x = ul_x + size
    br_y = ul_y + size

    # If completely outside
    if br_x <= 0 or br_y <= 0 or ul_x >= W or ul_y >= H:
        return hm

    # Clip hm region
    img_x0 = max(0, ul_x)
    img_y0 = max(0, ul_y)
    img_x1 = min(W, br_x)
    img_y1 = min(H, br_y)

    # Corresponding g region
    g_x0 = img_x0 - ul_x
    g_y0 = img_y0 - ul_y
    g_x1 = g_x0 + (img_x1 - img_x0)
    g_y1 = g_y0 + (img_y1 - img_y0)

    # Clip g region to be extra safe
    g_x0 = max(0, g_x0); g_y0 = max(0, g_y0)
    g_x1 = min(size, g_x1); g_y1 = min(size, g_y1)

    # Now take minimum overlap sizes
    patch_h = min(img_y1 - img_y0, g_y1 - g_y0)
    patch_w = min(img_x1 - img_x0, g_x1 - g_x0)

    if patch_h <= 0 or patch_w <= 0:
        return hm

    hm_patch = hm[img_y0:img_y0+patch_h, img_x0:img_x0+patch_w]
    g_patch  = g[g_y0:g_y0+patch_h, g_x0:g_x0+patch_w]

    hm[img_y0:img_y0+patch_h, img_x0:img_x0+patch_w] = np.maximum(hm_patch, g_patch)
    return hm

print("Patched draw_gaussian (fully robust).")

def generate_heatmaps(joints_xy_hm, joints_vis, heatmap_size, sigma):
    """
    heatmap_size: (Wm,Hm)
    returns heatmaps: (K,Hm,Wm), weights: (K,1)
    """
    Wm, Hm = heatmap_size
    heatmaps = np.zeros((K, Hm, Wm), dtype=np.float32)
    weights = joints_vis.astype(np.float32).reshape(K, 1)

    for i in range(K):
        if weights[i, 0] < 0.5:
            continue
        x, y = joints_xy_hm[i]
        if x < 0 or y < 0 or x >= Wm or y >= Hm:
            weights[i, 0] = 0.0
            continue
        heatmaps[i] = draw_gaussian(heatmaps[i], (x, y), sigma)

    return heatmaps, weights

print("Geometry helpers ready.")


**2.3**

Synthetic occlusion (cutout)

In [ ]:
def apply_synthetic_occlusion(img_bgr, p=0.5):
    if np.random.rand() > p:
        return img_bgr
    h, w, _ = img_bgr.shape
    occ_w = np.random.randint(max(1, w//8), max(2, w//3))
    occ_h = np.random.randint(max(1, h//8), max(2, h//3))
    x0 = np.random.randint(0, max(1, w - occ_w))
    y0 = np.random.randint(0, max(1, h - occ_h))
    img_bgr[y0:y0+occ_h, x0:x0+occ_w] = 0
    return img_bgr


**2.4**

Dataset class

Key design choices (match your formulation):

* Input size: (W,H) = (192,256)

* Heatmap size: (Wm,Hm) = (48,64) (stride=4)

* Visibility 𝑣𝑖: 1 only if COCO v==2 (visible), else 0

* Occluded joints get heatmap weight 0

In [ ]:
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import torch

class CocoKeypointsSinglePerson(Dataset):
    def __init__(
        self,
        img_dir,
        ann_file,
        is_train=True,
        input_size=(192, 256),      # (W,H)
        heatmap_size=(48, 64),      # (Wm,Hm) stride=4
        sigma=2.0,
        rot_factor=40,
        scale_factor=(0.7, 1.35),
        flip_prob=0.5,
        occ_prob=0.5,
    ):
        self.coco = COCO(ann_file)
        self.img_dir = img_dir
        self.is_train = is_train

        self.input_size = input_size
        self.heatmap_size = heatmap_size
        self.sigma = sigma

        self.rot_factor = rot_factor
        self.scale_factor = scale_factor
        self.flip_prob = flip_prob
        self.occ_prob = occ_prob

        self.ann_ids = []
        for ann_id, ann in self.coco.anns.items():
            if ann.get("iscrowd", 0) == 0 and ann.get("num_keypoints", 0) > 0:
                self.ann_ids.append(ann_id)

    def __len__(self):
        return len(self.ann_ids)

    def _xywh_to_center_scale(self, bbox):
        x, y, w, h = bbox
        center = np.array([x + w/2.0, y + h/2.0], dtype=np.float32)
        scale = max(w, h) * 1.25
        return center, scale

    def _flip_joints(self, joints, vis, img_w):
        joints[:, 0] = img_w - 1 - joints[:, 0]
        for a, b in COCO_FLIP_PAIRS:
            joints[[a, b]] = joints[[b, a]]
            vis[[a, b]] = vis[[b, a]]
        return joints, vis

    def __getitem__(self, idx):
        ann_id = self.ann_ids[idx]
        ann = self.coco.anns[ann_id]
        img_info = self.coco.loadImgs([ann["image_id"]])[0]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)

        # COCO keypoints: (K,3) [x,y,v], v: 0=not labeled, 1=labeled not visible, 2=visible
        kpts = np.array(ann["keypoints"], dtype=np.float32).reshape(K, 3)
        joints_xy = kpts[:, :2].copy()
        v_raw = kpts[:, 2].copy()

        # Our v_i in {0,1}: 1 only if visible (v==2)
        vis = (v_raw == 2).astype(np.float32)

        center, scale = self._xywh_to_center_scale(ann["bbox"])

        rot = 0.0
        if self.is_train:
            scale *= np.random.uniform(self.scale_factor[0], self.scale_factor[1])
            if np.random.rand() < 0.6:
                rot = np.random.uniform(-self.rot_factor, self.rot_factor)

        trans = get_affine_transform(center, scale, rot, self.input_size)

        in_w, in_h = self.input_size
        warped = cv2.warpAffine(img_bgr, trans, (in_w, in_h), flags=cv2.INTER_LINEAR)

        # Transform joints into input space
        joints_in = np.zeros((K, 2), dtype=np.float32)
        for i in range(K):
            if v_raw[i] > 0:  # labeled
                joints_in[i] = affine_transform(joints_xy[i], trans)
            else:
                joints_in[i] = 0.0

        # Flip
        if self.is_train and np.random.rand() < self.flip_prob:
            warped = warped[:, ::-1, :].copy()
            joints_in, vis = self._flip_joints(joints_in, vis, img_w=in_w)

        # Synthetic occlusion on image
        if self.is_train:
            warped = apply_synthetic_occlusion(warped, p=self.occ_prob)

        # Heatmap coords
        hm_w, hm_h = self.heatmap_size
        stride_x = in_w / hm_w
        stride_y = in_h / hm_h
        joints_hm = joints_in.copy()
        joints_hm[:, 0] /= stride_x
        joints_hm[:, 1] /= stride_y

        target_hm, target_weight = generate_heatmaps(joints_hm, vis, self.heatmap_size, self.sigma)

        # Image: RGB [0,1], CHW
        img_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = np.transpose(img_rgb, (2, 0, 1))

        return (
            torch.from_numpy(img),            # (3,H,W)
            torch.from_numpy(target_hm),      # (K,Hm,Wm)
            torch.from_numpy(target_weight),  # (K,1)
            torch.from_numpy(joints_in),      # (K,2) input coords
            torch.from_numpy(vis),            # (K,)
        )

print("Dataset class ready.")


**2.5**

Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader

train_ds = CocoKeypointsSinglePerson(
    img_dir=TRAIN_IMG_DIR, ann_file=TRAIN_ANN_FILE, is_train=True,
    input_size=(192,256), heatmap_size=(48,64), sigma=2.0, occ_prob=0.5
)
val_ds = CocoKeypointsSinglePerson(
    img_dir=VAL_IMG_DIR, ann_file=VAL_ANN_FILE, is_train=False,
    input_size=(192,256), heatmap_size=(48,64), sigma=2.0, occ_prob=0.0
)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))


**2.6**

Sanity visualization

In [ ]:
def show_one_sample(loader):
    img, hm, w, joints_in, vis = next(iter(loader))
    img0 = img[0].numpy().transpose(1,2,0)
    joints0 = joints_in[0].numpy()
    vis0 = vis[0].numpy()

    plt.figure(figsize=(5,6))
    plt.imshow(img0)
    for i in range(K):
        if vis0[i] > 0.5:
            plt.scatter(joints0[i,0], joints0[i,1], s=18)
            plt.text(joints0[i,0], joints0[i,1], str(i), fontsize=7)
    plt.title("Input crop + visible joints")
    plt.axis("off")
    plt.show()

    hm_sum = hm[0].numpy().sum(axis=0)  # (Hm,Wm)
    plt.figure(figsize=(5,4))
    plt.imshow(hm_sum)
    plt.title("Heatmap sum (64x48)")
    plt.axis("off")
    plt.show()

show_one_sample(train_loader)


# **Step 3**

**Model (from scratch, AMP-safe): backbone → heatmaps → soft-argmax → ROI feats → visibility logits**

Basic CNN backbone (stride=4)

Input: (B,3,256,192) → Feature map: (B,256,64,48)
This matches heatmap size (Hm,Wm)=(64,48).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SimpleStride4Backbone(nn.Module):
    """
    Input:  (B,3,H=256,W=192)
    Output: (B,C,H/4=64,W/4=48)
    """
    def __init__(self, out_channels=256):
        super().__init__()
        self.stem1 = ConvBNReLU(3, 64, k=3, s=2, p=1)     # -> 128x96
        self.stem2 = ConvBNReLU(64, 128, k=3, s=2, p=1)   # -> 64x48

        self.block1 = ConvBNReLU(128, 256, k=3, s=1, p=1)
        self.block2 = ConvBNReLU(256, 256, k=3, s=1, p=1)
        self.block3 = ConvBNReLU(256, out_channels, k=3, s=1, p=1)

    def forward(self, x):
        x = self.stem1(x)
        x = self.stem2(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x

print("Backbone ready.")


**3.2**

Heatmap head

Feature map (B,256,64,48) → Heatmaps (B,K,64,48)

In [ ]:
class HeatmapHead(nn.Module):
    def __init__(self, in_channels=256, num_joints=17):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels, k=3, s=1, p=1)
        self.out = nn.Conv2d(in_channels, num_joints, kernel_size=1, stride=1, padding=0)

    def forward(self, feat):
        return self.out(self.conv(feat))

print("Heatmap head ready.")


**3.3**

Soft-argmax (heatmaps → coords in heatmap space)

Returns coords (B,K,2) in (x,y) heatmap pixels.

In [ ]:
class SoftArgmax2D(nn.Module):
    def __init__(self, beta=80.0):
        super().__init__()
        self.beta = beta

    def forward(self, heatmaps):
        B, K_, H, W = heatmaps.shape
        hm = heatmaps.view(B, K_, -1)
        probs = F.softmax(hm * self.beta, dim=-1).view(B, K_, H, W)

        xs = torch.linspace(0, W-1, W, device=heatmaps.device, dtype=heatmaps.dtype)
        ys = torch.linspace(0, H-1, H, device=heatmaps.device, dtype=heatmaps.dtype)

        exp_x = (probs.sum(dim=2) * xs[None, None, :]).sum(dim=2)  # (B,K)
        exp_y = (probs.sum(dim=3) * ys[None, None, :]).sum(dim=2)  # (B,K)

        coords = torch.stack([exp_x, exp_y], dim=-1)  # (B,K,2)
        return coords, probs

print("SoftArgmax ready.")


**3.4**

ROI sampler (sample per-joint features at coords)

grid_sample gives (B,K,C) ROI features.

In [ ]:
class ROISampler(nn.Module):
    def forward(self, feat, coords_hm):
        """
        feat: (B,C,Hf,Wf)
        coords_hm: (B,K,2) in same resolution as feat (Hf,Wf)
        """
        B, C, Hf, Wf = feat.shape
        B2, K_, _ = coords_hm.shape
        assert B == B2

        x = coords_hm[..., 0]
        y = coords_hm[..., 1]

        x_norm = 2.0 * x / (Wf - 1) - 1.0
        y_norm = 2.0 * y / (Hf - 1) - 1.0

        grid = torch.stack([x_norm, y_norm], dim=-1)  # (B,K,2)
        grid = grid.view(B, K_, 1, 2)                 # (B,K,1,2)

        sampled = F.grid_sample(feat, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
        # sampled: (B,C,K,1) -> (B,K,C)
        sampled = sampled.squeeze(-1).permute(0, 2, 1).contiguous()
        return sampled

print("ROI sampler ready.")


**3.5**

Visibility head (logits) + full init module

AMP-safe design:

* Visibility head outputs logits

* compute vis_prob = sigmoid(logits) for gating in GNN later

In [ ]:
class VisibilityHead(nn.Module):
    def __init__(self, in_dim=256, hidden=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 1),
        )

    def forward(self, roi_feats):
        B, K_, C = roi_feats.shape
        logits = self.mlp(roi_feats.view(B*K_, C)).view(B, K_)
        return logits

class PoseInitModule(nn.Module):
    """
    Outputs:
      heatmaps:   (B,K,64,48)
      coords_hm:  (B,K,2) in heatmap coords
      coords_in:  (B,K,2) in input pixel coords
      roi_feats:  (B,K,256)
      vis_logits: (B,K)
      vis_prob:   (B,K)
    """
    def __init__(self, num_joints=17, feat_dim=256, beta=80.0, input_size=(192,256), heatmap_size=(48,64)):
        super().__init__()
        self.num_joints = num_joints
        self.feat_dim = feat_dim

        self.input_w, self.input_h = input_size      # (W,H)
        self.hm_w, self.hm_h = heatmap_size          # (Wm,Hm)

        self.backbone = SimpleStride4Backbone(out_channels=feat_dim)
        self.hm_head = HeatmapHead(in_channels=feat_dim, num_joints=num_joints)
        self.softargmax = SoftArgmax2D(beta=beta)
        self.roi = ROISampler()
        self.vis_head = VisibilityHead(in_dim=feat_dim, hidden=128)

        # stride from heatmap to input
        self.stride_x = self.input_w / self.hm_w
        self.stride_y = self.input_h / self.hm_h

    def forward(self, x):
        feat = self.backbone(x)             # (B,256,64,48)
        heatmaps = self.hm_head(feat)       # (B,K,64,48)

        coords_hm, hm_probs = self.softargmax(heatmaps)    # (B,K,2) in heatmap coords
        roi_feats = self.roi(feat, coords_hm)              # (B,K,256)

        vis_logits = self.vis_head(roi_feats)              # (B,K)
        vis_prob = torch.sigmoid(vis_logits)               # (B,K)

        coords_in = coords_hm.clone()
        coords_in[..., 0] = coords_in[..., 0] * self.stride_x
        coords_in[..., 1] = coords_in[..., 1] * self.stride_y

        return {
            "feat": feat,
            "heatmaps": heatmaps,
            "coords_hm": coords_hm,
            "coords_in": coords_in,
            "roi_feats": roi_feats,
            "vis_logits": vis_logits,
            "vis_prob": vis_prob,
            "hm_probs": hm_probs
        }

print("PoseInitModule ready (AMP-safe).")


**3.6**

Forward-pass sanity check

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_init = PoseInitModule(
    num_joints=K, feat_dim=256, beta=80.0,
    input_size=(192,256), heatmap_size=(48,64)
).to(device)

batch = next(iter(train_loader))
imgs = batch[0].to(device)

with torch.no_grad():
    out = model_init(imgs)

print("feat:", out["feat"].shape)
print("heatmaps:", out["heatmaps"].shape)
print("coords_hm:", out["coords_hm"].shape)
print("coords_in:", out["coords_in"].shape)
print("roi_feats:", out["roi_feats"].shape)
print("vis_logits:", out["vis_logits"].shape)
print("vis_prob:", out["vis_prob"].shape)


# **Step 4**



**GNN Pose Refiner (edge embeddings + message passing + visibility-aware update)**

Build directed edge list + edge_index

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_directed_edges(undirected_edges):
    directed = []
    for i, j in undirected_edges:
        directed.append((i, j))
        directed.append((j, i))
    return directed

DIRECTED_EDGES = make_directed_edges(COCO_EDGES)
E_dir = len(DIRECTED_EDGES)

edge_index = torch.tensor(DIRECTED_EDGES, dtype=torch.long).t().contiguous()  # (2, E_dir)

print("Num joints:", K)
print("Num undirected edges:", len(COCO_EDGES))
print("Num directed edges:", E_dir)
print("edge_index shape:", edge_index.shape)


**4.2**

MLP helper

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, act="gelu"):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.act = nn.GELU() if act == "gelu" else nn.ReLU(inplace=True)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


**4.3**

One GNN layer (AMP-safe index_add_)

Key fix included: m is created with dtype=msg_e.dtype, so AMP works.

In [ ]:
class GNNLayer(nn.Module):
    def __init__(self, hidden_dim=256, edge_dim=32):
        super().__init__()
        self.phi_m = MLP(hidden_dim + edge_dim, hidden_dim, hidden_dim, act="gelu")

        # update uses [h_i, m_i, v_i]
        self.fc_u = nn.Linear(hidden_dim * 2 + 1, hidden_dim)
        self.fc_g = nn.Linear(hidden_dim * 2 + 1, hidden_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h, vis_prob, edge_index, edge_embed):
        """
        h: (B,K,H)
        vis_prob: (B,K)
        edge_index: (2,E)
        edge_embed: (E,edge_dim)
        """
        B, K_, H = h.shape
        src = edge_index[0]  # (E,)
        dst = edge_index[1]  # (E,)

        h_src = h[:, src, :]  # (B,E,H)
        e = edge_embed.unsqueeze(0).expand(B, -1, -1)  # (B,E,edge_dim)

        msg_e = self.phi_m(torch.cat([h_src, e], dim=-1))  # (B,E,H)

        # AMP-safe aggregation
        m = torch.zeros((B, K_, H), device=h.device, dtype=msg_e.dtype)
        m.index_add_(1, dst, msg_e)

        v = vis_prob.unsqueeze(-1)  # (B,K,1)
        upd_in = torch.cat([h, m, v], dim=-1)

        delta = self.act(self.fc_u(upd_in))
        gate  = torch.sigmoid(self.fc_g(upd_in))

        h_new = h + gate * delta
        h_new = self.norm(h_new)
        return h_new


**4.4**

Full GraphPoseRefiner

Node embedding

Normalize coords before projection for stability.

In [ ]:
class GraphPoseRefiner(nn.Module):
    def __init__(
        self,
        num_joints=17,
        hidden_dim=256,
        roi_dim=256,
        edge_dim=32,
        num_layers=3,
        input_size=(192,256),   # (W,H)
        edge_index=edge_index
    ):
        super().__init__()
        self.num_joints = num_joints
        self.hidden_dim = hidden_dim
        self.roi_dim = roi_dim
        self.edge_dim = edge_dim
        self.num_layers = num_layers

        self.input_w, self.input_h = input_size

        # store edge_index as buffer
        self.register_buffer("edge_index", edge_index)

        # one embedding per directed edge
        self.edge_emb = nn.Embedding(edge_index.shape[1], edge_dim)

        # node projection: [coords_norm(2), vis(1), roi(256)] -> 256
        self.node_proj = nn.Linear(2 + 1 + roi_dim, hidden_dim)

        self.layers = nn.ModuleList([GNNLayer(hidden_dim, edge_dim) for _ in range(num_layers)])

        # regression head: predict delta (in input pixels)
        self.reg_head = nn.Linear(hidden_dim, 2)

    def forward(self, coords_in, roi_feats, vis_prob):
        """
        coords_in: (B,K,2) in input pixel coords
        roi_feats: (B,K,roi_dim)
        vis_prob:  (B,K)
        """
        B, K_, _ = coords_in.shape
        assert K_ == self.num_joints

        # normalize coords to [0,1] for embedding
        coords_norm = coords_in.clone()
        coords_norm[..., 0] = coords_norm[..., 0] / float(self.input_w)
        coords_norm[..., 1] = coords_norm[..., 1] / float(self.input_h)

        v = vis_prob.unsqueeze(-1)  # (B,K,1)
        h0 = torch.cat([coords_norm, v, roi_feats], dim=-1)  # (B,K,2+1+roi)
        h = self.node_proj(h0)                               # (B,K,256)

        edge_ids = torch.arange(self.edge_index.shape[1], device=h.device)
        e = self.edge_emb(edge_ids)  # (E,edge_dim)

        for layer in self.layers:
            h = layer(h, vis_prob, self.edge_index, e)

        delta = self.reg_head(h)           # (B,K,2)
        coords_refined = coords_in + delta # (B,K,2)

        return {
            "h_final": h,
            "delta": delta,
            "coords_refined": coords_refined
        }

print("GraphPoseRefiner ready.")


**4.5**

Sanity check: init module → GNN

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

gnn = GraphPoseRefiner(
    num_joints=K,
    hidden_dim=256,
    roi_dim=256,
    edge_dim=32,
    num_layers=3,
    input_size=(192,256),
    edge_index=edge_index
).to(device)

batch = next(iter(train_loader))
imgs = batch[0].to(device)

with torch.no_grad():
    out_init = model_init(imgs)
    out_gnn  = gnn(out_init["coords_in"], out_init["roi_feats"], out_init["vis_prob"])

print(device)
print("Init coords_in:", out_init["coords_in"].shape)
print("Refined coords:", out_gnn["coords_refined"].shape)
print("Delta:", out_gnn["delta"].shape)
print("h_final:", out_gnn["h_final"].shape)


# **Step 5**

**Losses + total objective (AMP-safe)**

Heatmap loss + coord loss + visibility loss (BCEWithLogits)

In [ ]:
import torch
import torch.nn.functional as F

def heatmap_mse_loss(pred_hm, target_hm, target_weight):
    """
    pred_hm: (B,K,Hm,Wm)
    target_hm: (B,K,Hm,Wm)
    target_weight: (B,K,1)
    """
    B, K_, Hm, Wm = pred_hm.shape
    w = target_weight.view(B, K_, 1, 1)
    loss = ((pred_hm - target_hm) ** 2) * w
    denom = w.sum() * Hm * Wm + 1e-6
    return loss.sum() / denom

def coord_mse_loss(coords_pred, coords_gt, vis_gt):
    """
    coords_pred: (B,K,2)
    coords_gt:   (B,K,2)
    vis_gt:      (B,K) supervise only visible joints
    """
    w = vis_gt.unsqueeze(-1)
    loss = ((coords_pred - coords_gt) ** 2) * w
    denom = w.sum() * 2.0 + 1e-6
    return loss.sum() / denom

def visibility_bce_logits_loss(vis_logits, vis_gt):
    """
    AMP-safe BCE with logits
    vis_logits: (B,K)
    vis_gt:     (B,K) in {0,1}
    """
    return F.binary_cross_entropy_with_logits(vis_logits, vis_gt)

print("Basic losses ready.")


**5.2**

Bone-length consistency loss

In [ ]:
def bone_length_loss(coords_pred, coords_gt, edges, labeled_mask=None):
    """
    coords_pred: (B,K,2)
    coords_gt:   (B,K,2)
    edges: list[(i,j)]
    labeled_mask: (B,K) optional; 1 if GT exists (v_raw>0). If None inferred from coords_gt != 0.
    """
    if labeled_mask is None:
        labeled_mask = ((coords_gt[..., 0] != 0) | (coords_gt[..., 1] != 0)).float()

    loss_sum = 0.0
    denom = 0.0

    for (i, j) in edges:
        m = labeled_mask[:, i] * labeled_mask[:, j]  # (B,)
        if m.sum() < 1:
            continue

        pred_len = torch.linalg.norm(coords_pred[:, i] - coords_pred[:, j], dim=-1)
        ref_len  = torch.linalg.norm(coords_gt[:, i]   - coords_gt[:, j], dim=-1)

        loss_sum += ((pred_len - ref_len) ** 2 * m).sum()
        denom += m.sum()

    return loss_sum / (denom + 1e-6)

print("Bone-length loss ready.")


**5.3**

Angle feasibility loss (soft penalty)

In [ ]:
import math

def angle_between(a, b, c):
    ba = a - b
    bc = c - b
    dot = (ba * bc).sum(dim=-1)
    n1 = torch.linalg.norm(ba, dim=-1)
    n2 = torch.linalg.norm(bc, dim=-1)
    cos = dot / (n1 * n2 + 1e-6)
    cos = torch.clamp(cos, -1.0, 1.0)
    return torch.acos(cos)

def range_penalty(theta, tmin, tmax):
    below = F.relu(tmin - theta)
    above = F.relu(theta - tmax)
    return below**2 + above**2

COCO_TRIPLETS = [
    (5, 7, 9),     # L elbow
    (6, 8, 10),    # R elbow
    (11, 13, 15),  # L knee
    (12, 14, 16),  # R knee
    (7, 5, 11),    # L shoulder (elbow-shoulder-hip)
    (8, 6, 12),    # R shoulder
    (5, 11, 13),   # L hip (shoulder-hip-knee)
    (6, 12, 14),   # R hip
]

ANGLE_RANGES = {t: (math.radians(5), math.radians(175)) for t in COCO_TRIPLETS}
# Slightly tighter for shoulder/hip proxy angles
for t in [(7,5,11),(8,6,12),(5,11,13),(6,12,14)]:
    ANGLE_RANGES[t] = (math.radians(10), math.radians(170))

def angle_feasibility_loss(coords_pred, triplets=COCO_TRIPLETS, angle_ranges=ANGLE_RANGES, labeled_mask=None):
    B, K_, _ = coords_pred.shape
    if labeled_mask is None:
        labeled_mask = torch.ones((B, K_), device=coords_pred.device, dtype=coords_pred.dtype)

    loss_sum = 0.0
    denom = 0.0

    for (i, j, k) in triplets:
        m = labeled_mask[:, i] * labeled_mask[:, j] * labeled_mask[:, k]
        if m.sum() < 1:
            continue
        tmin, tmax = angle_ranges[(i, j, k)]
        theta = angle_between(coords_pred[:, i], coords_pred[:, j], coords_pred[:, k])
        p = range_penalty(theta, tmin, tmax)
        loss_sum += (p * m).sum()
        denom += m.sum()

    return loss_sum / (denom + 1e-6)

print("Angle loss ready.")


**5.4**

Total loss function (matches λs):

* heatmap loss (initial estimation)

* visibility loss (logits)

* GNN coord loss (visible joints)

* bone + angle regularizers

* λ1=0.4, λ2=0.2, λ3=1.0

In [ ]:
LAMBDA_BONE  = 0.4
LAMBDA_ANGLE = 0.2
LAMBDA_GNN   = 1.0

def compute_total_loss(model_init, gnn, batch, device):
    imgs, target_hm, target_w, joints_in_gt, vis_gt = batch
    imgs = imgs.to(device)
    target_hm = target_hm.to(device)
    target_w  = target_w.to(device)
    joints_in_gt = joints_in_gt.to(device)
    vis_gt = vis_gt.to(device)

    # labeled mask: GT exists if not (0,0)
    labeled_mask = ((joints_in_gt[..., 0] != 0) | (joints_in_gt[..., 1] != 0)).float()

    out_init = model_init(imgs)
    out_gnn  = gnn(out_init["coords_in"], out_init["roi_feats"], out_init["vis_prob"])

    loss_hm   = heatmap_mse_loss(out_init["heatmaps"], target_hm, target_w)
    loss_vis  = visibility_bce_logits_loss(out_init["vis_logits"], vis_gt)
    loss_gnn  = coord_mse_loss(out_gnn["coords_refined"], joints_in_gt, vis_gt)

    loss_bone = bone_length_loss(out_gnn["coords_refined"], joints_in_gt, COCO_EDGES, labeled_mask=labeled_mask)
    loss_ang  = angle_feasibility_loss(out_gnn["coords_refined"], labeled_mask=labeled_mask)

    loss_total = (
        loss_hm
        + loss_vis
        + LAMBDA_BONE  * loss_bone
        + LAMBDA_ANGLE * loss_ang
        + LAMBDA_GNN   * loss_gnn
    )

    logs = {
        "total": loss_total.detach().item(),
        "hm":    loss_hm.detach().item(),
        "vis":   loss_vis.detach().item(),
        "gnn":   loss_gnn.detach().item(),
        "bone":  loss_bone.detach().item(),
        "angle": loss_ang.detach().item(),
    }
    return loss_total, logs

print("Total loss ready.")


**5.5**

Sanity check (one batch loss)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

batch = next(iter(train_loader))
loss_total, logs = compute_total_loss(model_init, gnn, batch, device)
print("Loss logs:", logs)


# **Step 6**

**Training loop (fresh, AMP-safe, no warnings)**

Optimizer + warmup+cosine LR scheduler + AMP setup

In [ ]:
import math
from torch.amp import autocast, GradScaler

device = "cuda" if torch.cuda.is_available() else "cpu"
model_init = model_init.to(device)
gnn = gnn.to(device)

# Optimizer (both modules)
params = list(model_init.parameters()) + list(gnn.parameters())
optimizer = torch.optim.Adam(params, lr=1e-4)

# Schedule settings (matches your protocol style)
BASE_LR = 1e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 5
TOTAL_EPOCHS = 2  # sanity run; set to 220 later

def lr_scale(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    t = (epoch - WARMUP_EPOCHS) / max(1, (TOTAL_EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1.0 + math.cos(math.pi * t))

def set_lr(optimizer, epoch):
    s = lr_scale(epoch)
    lr = MIN_LR + (BASE_LR - MIN_LR) * s
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    return lr

scaler = GradScaler(enabled=(device == "cuda"))
print("Setup done. AMP enabled:", scaler.is_enabled())


**6.2**

Train one epoch

In [ ]:
from tqdm import tqdm
import torch

def train_one_epoch(model_init, gnn, loader, optimizer, epoch, grad_clip=1.0):
    model_init.train()
    gnn.train()

    lr = set_lr(optimizer, epoch)
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS} lr={lr:.2e}")

    running = {"total":0, "hm":0, "vis":0, "gnn":0, "bone":0, "angle":0}
    n = 0

    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=(device == "cuda")):
            loss, logs = compute_total_loss(model_init, gnn, batch, device)

        scaler.scale(loss).backward()

        if grad_clip is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(model_init.parameters()) + list(gnn.parameters()),
                grad_clip
            )

        scaler.step(optimizer)
        scaler.update()

        n += 1
        for k in running:
            running[k] += logs[k]

        avg = {k: running[k]/n for k in running}
        pbar.set_postfix(total=f"{avg['total']:.1f}", gnn=f"{avg['gnn']:.1f}", bone=f"{avg['bone']:.1f}")

    return {k: running[k]/n for k in running}

**6.3**

Eval one epoch

In [ ]:
@torch.no_grad()
def eval_one_epoch(model_init, gnn, loader):
    model_init.eval()
    gnn.eval()

    running = {"total":0, "hm":0, "vis":0, "gnn":0, "bone":0, "angle":0}
    n = 0

    for batch in loader:
        with autocast(device_type="cuda", enabled=(device == "cuda")):
            loss, logs = compute_total_loss(model_init, gnn, batch, device)
        n += 1
        for k in running:
            running[k] += logs[k]

    return {k: running[k]/n for k in running}


**6.4**

Run training (Check nuber of Epochs)

In [ ]:
for epoch in range(TOTAL_EPOCHS):
    train_logs = train_one_epoch(model_init, gnn, train_loader, optimizer, epoch, grad_clip=1.0)
    val_logs   = eval_one_epoch(model_init, gnn, val_loader)

    print(f"\nEpoch {epoch+1}/{TOTAL_EPOCHS}")
    print("  Train:", {k: round(v, 4) for k,v in train_logs.items()})
    print("  Val:  ", {k: round(v, 4) for k,v in val_logs.items()})


**6.5**

Overfit 1 batch (quick correctness test)

If this doesn’t reduce loss, something is wrong. It should drop noticeably within ~50–200 steps.

In [ ]:
# Take one batch and overfit
batch0 = next(iter(train_loader))

for p in model_init.parameters(): p.requires_grad = True
for p in gnn.parameters(): p.requires_grad = True

optimizer = torch.optim.Adam(list(model_init.parameters()) + list(gnn.parameters()), lr=1e-4)
scaler = GradScaler(enabled=(device == "cuda"))

for step in range(100):
    optimizer.zero_grad(set_to_none=True)
    with autocast(device_type="cuda", enabled=(device == "cuda")):
        loss, logs = compute_total_loss(model_init, gnn, batch0, device)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % 10 == 0:
        print(f"step {step:03d} | total={logs['total']:.2f} gnn={logs['gnn']:.2f} bone={logs['bone']:.2f}")


# **Step 7**

**Inference + Visualization (predicted pose + GT + visibility)**

This will:

* take a batch from val_loader

* run model_init + gnn

* plot GT visible joints vs pred refined joints

* draw skeleton edges

* print per-joint visibility probability 𝑣^𝑖
	​


Drawing utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def draw_skeleton(ax, joints_xy, edges, color="lime", lw=2, alpha=0.9):
    """
    joints_xy: (K,2) in input pixel coords
    """
    for (i, j) in edges:
        xi, yi = joints_xy[i]
        xj, yj = joints_xy[j]
        ax.plot([xi, xj], [yi, yj], linewidth=lw, alpha=alpha, color=color)

def draw_joints(ax, joints_xy, vis_mask=None, color="red", size=20, alpha=0.9, annotate=False):
    """
    vis_mask: (K,) in {0,1}, if provided draw only visible.
    """
    K_ = joints_xy.shape[0]
    for i in range(K_):
        if vis_mask is not None and vis_mask[i] < 0.5:
            continue
        x, y = joints_xy[i]
        ax.scatter([x], [y], s=size, color=color, alpha=alpha)
        if annotate:
            ax.text(x, y, str(i), fontsize=7, color=color)

print("Drawing utils ready.")


**7.2**

Run inference on one validation batch

In [ ]:
import torch
from torch.amp import autocast

device = "cuda" if torch.cuda.is_available() else "cpu"
model_init.eval()
gnn.eval()

batch = next(iter(val_loader))
imgs, target_hm, target_w, joints_in_gt, vis_gt = batch

imgs = imgs.to(device)

with torch.no_grad():
    with autocast(device_type="cuda", enabled=(device == "cuda")):
        out_init = model_init(imgs)
        out_gnn  = gnn(out_init["coords_in"], out_init["roi_feats"], out_init["vis_prob"])

# Move outputs to CPU for plotting
img0 = imgs[0].detach().cpu().numpy().transpose(1,2,0)  # (H,W,3)
gt0 = joints_in_gt[0].numpy()                           # (K,2)
vis0 = vis_gt[0].numpy()                                # (K,)
pred0 = out_gnn["coords_refined"][0].detach().cpu().numpy()  # (K,2)
visprob0 = out_init["vis_prob"][0].detach().cpu().numpy()    # (K,)

print("Shapes:", img0.shape, gt0.shape, pred0.shape, visprob0.shape)
print("Visible GT joints in sample:", int(vis0.sum()))


**7.3**

Visualization: GT vs Pred

In [ ]:
# Clamp predictions to image bounds (just for visualization neatness)
H, W = img0.shape[:2]
pred0_clamped = pred0.copy()
pred0_clamped[:, 0] = np.clip(pred0_clamped[:, 0], 0, W-1)
pred0_clamped[:, 1] = np.clip(pred0_clamped[:, 1], 0, H-1)

plt.figure(figsize=(12,5))

# --- Panel 1: GT visible joints ---
ax1 = plt.subplot(1,2,1)
ax1.imshow(img0)
draw_skeleton(ax1, gt0, COCO_EDGES, color="cyan", lw=2, alpha=0.9)
draw_joints(ax1, gt0, vis_mask=vis0, color="yellow", size=25, alpha=0.95, annotate=True)
ax1.set_title("GT (visible joints only)")
ax1.axis("off")

# --- Panel 2: Pred refined joints ---
ax2 = plt.subplot(1,2,2)
ax2.imshow(img0)
draw_skeleton(ax2, pred0_clamped, COCO_EDGES, color="lime", lw=2, alpha=0.9)
draw_joints(ax2, pred0_clamped, vis_mask=None, color="red", size=18, alpha=0.85, annotate=False)
# overlay GT visible for reference
draw_joints(ax2, gt0, vis_mask=vis0, color="yellow", size=25, alpha=0.95, annotate=False)
ax2.set_title("Pred refined (red) + GT visible (yellow)")
ax2.axis("off")

plt.show()


**7.4**

Visibility probabilities per joint

In [ ]:
# Print a clean per-joint visibility report
order = list(range(K))
print("Joint visibility (GT vs predicted):")
for i in order:
    name = COCO_KPT_NAMES[i]
    print(f"{i:02d} {name:>14s} | GT_vis={int(vis0[i])} | pred_vis_prob={visprob0[i]:.3f}")


# **Step 8**

**Evaluation metric (PCK@0.05 / 0.1) on val2017**

COCO’s official metric is OKS/AP, but implementing full COCOeval for keypoints is longer. For a clean next step, we’ll compute PCK (Percentage of Correct Keypoints) normalized by person scale (bbox max side). This is standard for debugging and progress tracking.

* PCK@0.05 and PCK@0.10 over visible joints only

* Runs over a subset or whole val_loader

* Lets you track improvements during training

Helper: per-sample scale + PCK computation

In [ ]:
import numpy as np
import torch
from torch.amp import autocast

@torch.no_grad()
def evaluate_pck(model_init, gnn, loader, device, max_batches=None, thresholds=(0.05, 0.10)):
    model_init.eval()
    gnn.eval()

    correct = {t: 0 for t in thresholds}
    total = 0

    for b_idx, batch in enumerate(loader):
        if max_batches is not None and b_idx >= max_batches:
            break

        imgs, _, _, joints_gt, vis_gt = batch
        imgs = imgs.to(device)
        joints_gt = joints_gt.to(device)     # (B,K,2)
        vis_gt = vis_gt.to(device)           # (B,K)

        # scale proxy: bbox max side from GT labeled joints (robust enough for PCK)
        labeled = ((joints_gt[...,0]!=0) | (joints_gt[...,1]!=0)).float()  # (B,K)
        # min/max over labeled joints
        big = 1e9
        x = joints_gt[...,0] + (1.0 - labeled) * big
        y = joints_gt[...,1] + (1.0 - labeled) * big
        x_min = x.min(dim=1).values
        y_min = y.min(dim=1).values

        x2 = joints_gt[...,0] - (1.0 - labeled) * big
        y2 = joints_gt[...,1] - (1.0 - labeled) * big
        x_max = x2.max(dim=1).values
        y_max = y2.max(dim=1).values

        scale = torch.maximum(x_max - x_min, y_max - y_min)  # (B,)
        scale = torch.clamp(scale, min=1.0)

        with autocast(device_type="cuda", enabled=(device == "cuda")):
            out_init = model_init(imgs)
            out_gnn = gnn(out_init["coords_in"], out_init["roi_feats"], out_init["vis_prob"])
            pred = out_gnn["coords_refined"]  # (B,K,2)

        # distances
        d = torch.linalg.norm(pred - joints_gt, dim=-1)  # (B,K)
        # normalize by scale
        d_norm = d / scale.unsqueeze(-1)

        # count only visible GT joints
        vis = (vis_gt > 0.5)

        for t in thresholds:
            correct[t] += ((d_norm < t) & vis).sum().item()

        total += vis.sum().item()

    pck = {f"PCK@{t:.2f}": (correct[t] / max(1, total)) for t in thresholds}
    return pck, total

print("PCK evaluator ready.")


**8.2**

Run evaluation (fast: first 50 batches, Can be text on large batches)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pck, total = evaluate_pck(model_init, gnn, val_loader, device, max_batches=50, thresholds=(0.05, 0.10))
print("Evaluated visible joints:", total)
print(pck)

# **Step 9**

**COCO Keypoint AP (OKS) using COCOeval**

**Add a VAL dataset that also returns the inverse affine transform + meta**

In [ ]:
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
import torch

class CocoKeypointsSinglePersonEval(Dataset):
    """
    Like your dataset, but returns meta to map predictions back to original image coords.
    """
    def __init__(self, img_dir, ann_file, input_size=(192,256)):
        self.coco = COCO(ann_file)
        self.img_dir = img_dir
        self.input_size = input_size  # (W,H)

        self.ann_ids = []
        for ann_id, ann in self.coco.anns.items():
            if ann.get("iscrowd", 0) == 0 and ann.get("num_keypoints", 0) > 0:
                self.ann_ids.append(ann_id)

    def __len__(self):
        return len(self.ann_ids)

    def _xywh_to_center_scale(self, bbox):
        x, y, w, h = bbox
        center = np.array([x + w/2.0, y + h/2.0], dtype=np.float32)
        scale = max(w, h) * 1.25
        return center, scale

    def __getitem__(self, idx):
        ann_id = self.ann_ids[idx]
        ann = self.coco.anns[ann_id]
        img_info = self.coco.loadImgs([ann["image_id"]])[0]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(img_path)

        kpts = np.array(ann["keypoints"], dtype=np.float32).reshape(K, 3)
        joints_xy = kpts[:, :2].copy()
        v_raw = kpts[:, 2].copy()

        # visible joints only (your formulation)
        vis = (v_raw == 2).astype(np.float32)

        center, scale = self._xywh_to_center_scale(ann["bbox"])

        rot = 0.0  # IMPORTANT: no random aug in eval
        trans = get_affine_transform(center, scale, rot, self.input_size)     # orig -> input
        inv_trans = cv2.invertAffineTransform(trans)                          # input -> orig

        in_w, in_h = self.input_size
        warped = cv2.warpAffine(img_bgr, trans, (in_w, in_h), flags=cv2.INTER_LINEAR)

        # image tensor
        img_rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = np.transpose(img_rgb, (2,0,1))

        meta = {
            "image_id": int(ann["image_id"]),
            "ann_id": int(ann_id),
            "bbox": ann["bbox"],          # [x,y,w,h]
            "inv_trans": inv_trans.astype(np.float32),  # 2x3
        }

        return torch.from_numpy(img), torch.from_numpy(vis), meta

print("Eval dataset class ready.")


**9.2**

Create eval loader (no shuffling)

In [ ]:
eval_ds = CocoKeypointsSinglePersonEval(VAL_IMG_DIR, VAL_ANN_FILE, input_size=(192,256))
eval_loader = DataLoader(eval_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Eval samples:", len(eval_ds))


**9.3**

Build COCO-format predictions (map back to original image coords)

In [ ]:
import json
from torch.amp import autocast

def apply_inv_affine(coords_in, inv_trans):
    """
    coords_in: (K,2) in input pixels
    inv_trans: (2,3) maps input -> original
    returns: (K,2) in original pixels
    """
    K_ = coords_in.shape[0]
    ones = np.ones((K_, 1), dtype=np.float32)
    pts = np.concatenate([coords_in.astype(np.float32), ones], axis=1)  # (K,3)
    out = (inv_trans @ pts.T).T  # (K,2)
    return out

@torch.no_grad()
def build_coco_keypoint_results_fixed(model_init, gnn, loader, device, max_batches=None):
    model_init.eval()
    gnn.eval()

    results = []
    for b_idx, (imgs, vis_gt, meta) in enumerate(loader):
        if max_batches is not None and b_idx >= max_batches:
            break

        imgs = imgs.to(device)

        with autocast(device_type="cuda", enabled=(device=="cuda")):
            out_init = model_init(imgs)
            out_gnn  = gnn(out_init["coords_in"], out_init["roi_feats"], out_init["vis_prob"])

        coords_in = out_gnn["coords_refined"].detach().cpu().numpy()    # (B,K,2)

        image_ids = meta["image_id"]
        inv_trans_batch = meta["inv_trans"].numpy() if isinstance(meta["inv_trans"], torch.Tensor) else np.array(meta["inv_trans"])

        B = coords_in.shape[0]
        for n in range(B):
            inv_trans = inv_trans_batch[n]  # (2,3)
            kp_orig = apply_inv_affine(coords_in[n], inv_trans)  # (K,2)

            kps = []
            for i in range(K):
                x, y = kp_orig[i]
                kps.extend([float(x), float(y), 1.0])  # keypoint score = 1.0

            results.append({
                "image_id": int(image_ids[n]),
                "category_id": 1,
                "keypoints": kps,
                "score": 1.0,  # instance score = 1.0
            })

    return results


device = "cuda" if torch.cuda.is_available() else "cpu"
coco_results = build_coco_keypoint_results_fixed(model_init, gnn, eval_loader, device, max_batches=None)
print("Built detections:", len(coco_results))

#coco_results = build_coco_keypoint_results(model_init, gnn, eval_loader, device, max_batches=None)
print("Built detections:", len(coco_results))
print("Example:", coco_results[0].keys())


**9.4**

Run COCOeval keypoints AP (OKS)

In [ ]:
from pycocotools.cocoeval import COCOeval

# Save results to json
pred_path = "/content/occlusion_pose_gnn/outputs/keypoints_results.json"
with open(pred_path, "w") as f:
    json.dump(coco_results, f)

# COCOeval
cocoGt = COCO(VAL_ANN_FILE)
cocoDt = cocoGt.loadRes(pred_path)

cocoEval = COCOeval(cocoGt, cocoDt, iouType="keypoints")
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()
